In [1]:
import torch
from torch import nn
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from dateutil.relativedelta import relativedelta

In [2]:
df = pd.read_csv("../../../Data/Final/Imputation/changi_imp_final.csv")
print(df.head())

            x         y          Date      Value
0  103.964566  1.350459  Mar-Apr 2000  23.775064
1  103.964566  1.351269  Mar-Apr 2000  23.403744
2  103.964566  1.352079  Mar-Apr 2000  22.966872
3  103.964566  1.352889  Mar-Apr 2000  22.417757
4  103.965377  1.348838  Mar-Apr 2000  23.699749


In [3]:
## THIS CODE IS TO REFORMAT THE TIME INDEX IN THE DATASET ##

# Function to create a time index
def create_time_index(df):
    bimonthly_map = {"Jan-Feb": 1, "Mar-Apr": 2, "May-Jun": 3, "Jul-Aug": 4, "Sep-Oct": 5, "Nov-Dec": 6}
    
    df["Year"] = df["Date"].str[-4:].astype(int)  # Extract year
    df["Period"] = df["Date"].str[:-5].map(bimonthly_map)  # Extract period and map it

    df["time_index"] = (df["Year"] - 2000) * 6 + df["Period"]  # Compute time index
    df = df.drop(columns=["Year", "Period"])  # Drop extra columns
    
    return df

df = create_time_index(df)

### Window size = 9 years

In [4]:
## THIS CODE IS TO EXTRACT ALL POSSIBLE WINDOWS OF SIZE = 9 ##

# Define sequence length
sequence_length = 9 * 6  # 54 time steps (input)
target_length = 12  # Predict the next 12 time steps

# Group by (x, y) and process each time series separately
grouped = df.groupby(["x", "y"])

# Lists to store input sequences and target sequences separately
input_sequences = []
target_sequences = []
locations = []

# Iterate over each coordinate group
for (x, y), group in grouped:
    # Sort by time index
    group = group.sort_values(by="time_index")

    # Extract LST values
    values = group["Value"].values

    # Generate sequences
    for i in range(len(values) - sequence_length - target_length + 1):
        input_seq = values[i : i + sequence_length]  # Past 54 values
        target_seq = values[i + sequence_length : i + sequence_length + target_length]  # Next 12 values
        
        # Store sequences separately
        input_sequences.append([x, y] + list(input_seq))
        target_sequences.append([x, y] + list(target_seq))
        locations.append([x, y])

# Define column names
input_columns = ["x", "y"] + [f"LST_t-{i}" for i in range(sequence_length, 0, -1)]
target_columns = ["x", "y"] + [f"Target_t+{i}" for i in range(1, target_length + 1)]

# Convert to DataFrames
input_df = pd.DataFrame(input_sequences, columns=input_columns)
target_df = pd.DataFrame(target_sequences, columns=target_columns)

'''
# Save input and target sequences as separate CSV files
input_df.to_csv("changi_inputs_9.csv", index=False)
target_df.to_csv("changi_targets_10.csv", index=False)
'''

# Print sample sequences
print("Input sequences sample for window = 9:")
print(input_df.head())
print("\nTarget sequences sample for window = 9:")
print(target_df.head())

Input sequences sample for window = 9:
            x         y   LST_t-54   LST_t-53   LST_t-52   LST_t-51  \
0  103.964566  1.350459  23.775064  27.537897  27.519318  32.314507   
1  103.964566  1.350459  27.537897  27.519318  32.314507  23.391225   
2  103.964566  1.350459  27.519318  32.314507  23.391225  29.400195   
3  103.964566  1.350459  32.314507  23.391225  29.400195  29.712163   
4  103.964566  1.350459  23.391225  29.400195  29.712163  27.845558   

    LST_t-50   LST_t-49   LST_t-48   LST_t-47  ...   LST_t-10    LST_t-9  \
0  23.391225  29.400195  29.712163  27.845558  ...  19.414645  30.540039   
1  29.400195  29.712163  27.845558  24.015326  ...  30.540039  24.220239   
2  29.712163  27.845558  24.015326  25.797058  ...  24.220239  24.799115   
3  27.845558  24.015326  25.797058  19.449094  ...  24.799115  17.476492   
4  24.015326  25.797058  19.449094  29.891817  ...  17.476492  25.289893   

     LST_t-8    LST_t-7    LST_t-6    LST_t-5    LST_t-4    LST_t-3  \
0  24.

In [5]:
## THIS CODE IS TO RANDOMLY SAMPLE 10 WINDOWS FOR FORECASTING ##

np.random.seed(5188)  # For reproducibility

# Ensure both DataFrames have the same indices
assert len(input_df) == len(target_df), "Mismatch in input and target sizes"

# Randomly sample 10 indices

sample_indices = np.random.choice(len(input_df), 10, replace=False)

# Get sampled input and target sequences
sampled_inputs = input_df.iloc[sample_indices].reset_index(drop=True)
sampled_targets = target_df.iloc[sample_indices].reset_index(drop=True)

'''
# Save to CSV
sampled_inputs.to_csv("sampled_inputs.csv", index=False)
sampled_targets.to_csv("sampled_targets.csv", index=False)

'''

# Print sample
print(sampled_inputs.head())
print(sampled_targets.head())

            x         y   LST_t-54   LST_t-53   LST_t-52   LST_t-51  \
0  103.989682  1.356130  37.598484  19.442255  36.787585  29.064563   
1  104.006695  1.343977  31.303300  20.439904  31.145982  19.107546   
2  104.013177  1.368283  29.145823  35.510535  35.293890  23.156479   
3  104.011557  1.319672  30.431485  19.573795  30.070998  20.470991   
4  104.020468  1.367472  30.221030  27.868137  27.683563  19.795960   

    LST_t-50   LST_t-49   LST_t-48   LST_t-47  ...   LST_t-10    LST_t-9  \
0  24.668327  23.602310  23.674132  24.704418  ...  32.837948  25.144627   
1  25.238522  22.212594  25.889966  20.839769  ...  25.298094  22.326469   
2  24.207665  22.722342  31.501010  26.823871  ...  26.803807  22.557500   
3  26.157075  24.810787  24.596404  24.604110  ...  25.681257  24.510559   
4  26.476798  26.465944  24.715585  25.165858  ...  24.889567  25.340550   

     LST_t-8    LST_t-7    LST_t-6    LST_t-5    LST_t-4    LST_t-3  \
0  30.066594  34.736535  26.374424  29.253744

In [6]:
## THIS CODE IS TO CREATE THE TENSORS FOR TRAINING ##

# Z-score normalization: we normalize each feature individually
# Normalize inputs (past 54 time steps)
X = sampled_inputs.iloc[:, 2:].values  # Exclude 'x' and 'y' columns
y = sampled_targets.iloc[:, 2:].values  # Exclude 'x' and 'y' columns

# Calculate mean and standard deviation for each feature in the input (X) and target (y)
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
y_mean = y.mean(axis=0)
y_std = y.std(axis=0)

# Normalize inputs and targets
X_normalized = (X - X_mean) / X_std
y_normalized = (y - y_mean) / y_std

# Convert to PyTorch tensors
X_tensor = torch.tensor(X_normalized, dtype=torch.float32)
y_tensor = torch.tensor(y_normalized, dtype=torch.float32)

# Reshape the input tensor for LSTM: (batch_size, sequence_length, input_dim)
# Here, each sample has 54 time steps (sequence_length) and 1 feature per time step (input_dim).
X_tensor = X_tensor.view(X_tensor.shape[0], X_tensor.shape[1], 1)  # 1 feature per time step

# Print tensor shapes for verification
print(f"Input tensor shape: {X_tensor.shape}")
print(f"Target tensor shape: {y_tensor.shape}")

# Check the first sample
print(X_tensor[0])
print(y_tensor[0])

Input tensor shape: torch.Size([10, 54, 1])
Target tensor shape: torch.Size([10, 12])
tensor([[ 1.6174],
        [-0.8563],
        [ 1.4942],
        [ 1.8701],
        [-0.1560],
        [-0.3225],
        [-0.7993],
        [ 0.1466],
        [ 0.1845],
        [-0.6975],
        [-1.0409],
        [ 0.9967],
        [ 0.9972],
        [ 2.0665],
        [-0.5601],
        [-1.3567],
        [ 2.0677],
        [ 1.5046],
        [ 1.8236],
        [ 0.4564],
        [-0.0742],
        [-0.6002],
        [ 1.8996],
        [-1.0564],
        [ 1.5133],
        [-0.4828],
        [ 1.5195],
        [ 1.1104],
        [ 0.0508],
        [ 0.7011],
        [ 1.3201],
        [ 1.7340],
        [ 1.1810],
        [-0.9681],
        [ 1.1952],
        [ 1.2644],
        [ 1.3627],
        [ 0.9895],
        [-0.0409],
        [ 0.4448],
        [ 0.6668],
        [ 1.0037],
        [-0.1235],
        [ 0.6905],
        [ 1.9924],
        [ 0.5171],
        [ 1.6698],
        [ 1.7072],
  

In [7]:
## THIS CODE IS TO CREATE THE LSTM ##

class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=12, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, output_size)  # Fully connected layer for 12-step output

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  # Get LSTM output
        last_time_step = lstm_out[:, -1, :]  # Get last output of the sequence
        output = self.fc(last_time_step)  # Pass through FC layer
        return output

In [8]:
## THIS CODE IS TO TRAIN THE LSTM ##

# Instantiate the model
model = LSTMModel()

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    outputs = model(X_tensor)
    loss = criterion(outputs, y_tensor)
    
    # Backward pass and optimization
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [10/100], Loss: 0.9902
Epoch [20/100], Loss: 0.9227
Epoch [30/100], Loss: 0.6964
Epoch [40/100], Loss: 0.6248
Epoch [50/100], Loss: 0.5983
Epoch [60/100], Loss: 0.5654
Epoch [70/100], Loss: 0.5280
Epoch [80/100], Loss: 0.4597
Epoch [90/100], Loss: 0.3590
Epoch [100/100], Loss: 0.2606


In [9]:
# Evaluate model
model.eval()  # Set the model to evaluation mode
with torch.no_grad():  # Disable gradient calculation for evaluation
    predictions = model(X_tensor)  # Predict the next 12 steps (future)

# Denormalize the predictions back to the original scale
predictions_denorm = predictions.numpy() * y_std + y_mean

# Denormalize the targets back to the original scale
y_denorm = y_tensor.numpy() * y_std + y_mean

# Create the DataFrame to store the actual predicted values
# The prediction columns correspond to the future time steps (12 columns)
pred_df_9 = pd.DataFrame(predictions_denorm, columns=sampled_targets.columns[2:])  # Using 12 future columns from target_df

# Insert x and y coordinates into the dataframe
pred_df_9.insert(0, "x", sampled_inputs["x"])
pred_df_9.insert(1, "y", sampled_inputs["y"])

# Save the DataFrame to a CSV file
pred_df_9.to_csv("pred_9.csv", index=False)

# Print the DataFrame to inspect
print(pred_df_9)

            x         y  Target_t+1  Target_t+2  Target_t+3  Target_t+4  \
0  103.989682  1.356130   25.597072   25.498780   25.669846   25.074753   
1  104.006695  1.343977   24.194678   26.612755   24.271583   25.852698   
2  104.013177  1.368283   30.980618   19.341739   31.341805   22.280645   
3  104.011557  1.319672   22.791336   24.389847   22.565258   23.531185   
4  104.020468  1.367472   24.114594   22.023696   22.298047   22.605890   
5  104.031001  1.358560   26.180889   24.010422   24.537507   24.184826   
6  103.982390  1.393398   24.506476   20.502118   22.067488   22.113369   
7  103.990492  1.331014   22.664125   25.708641   22.807546   24.855930   
8  103.976719  1.352079   24.417254   26.249170   24.438976   25.689911   
9  103.990492  1.346408   26.158782   24.126420   27.303558   24.127443   

   Target_t+5  Target_t+6  Target_t+7  Target_t+8  Target_t+9  Target_t+10  \
0   22.626798   30.856375   26.070553   27.758252   28.413739    22.079900   
1   24.249401   28

In [10]:
#print(sampled_targets.columns)
#print(pred_df_9.columns)

In [12]:
## THIS CODE IS TO CALCULATE RMSE FOR EACH TIME STEP ##

#assert (pred_df_9[['x', 'y']].equals(target_df_9[['x', 'y']])), "x, y coordinates do not match."

# Define the forecast time steps
forecast_steps = [1, 3, 6, 9, 12]

# Initialize an empty list to store the RMSE values for each (x, y)
rmse_list = []

# Iterate through each (x, y) pair
for idx, row in pred_df_9.iterrows():
    x = row['x']
    y = row['y']
    
    # Extract the predicted values (from t+1 to t+12)
    predicted_values = row[2:].values  # The predicted columns (from 'Target_t+1' to 'Target_t+12')
    
    # Extract the corresponding true values (from t+1 to t+12)
    true_values = target_df.iloc[idx, 2:].values  # The target columns (from 'Target_t+1' to 'Target_t+12')
    
    # Calculate RMSE for each forecast step (f = 1, f = 3, f = 6, f = 9, f = 12)
    rmse_values = [
        np.sqrt(np.mean((predicted_values[step - 1] - true_values[step - 1]) ** 2))
        for step in forecast_steps
    ]
    
    # Calculate the average RMSE for the entire forecast period (t+1 to t+12)
    avg_rmse = np.sqrt(np.mean((predicted_values - true_values) ** 2))
    
    # Append the RMSE values for this (x, y) pair along with the average RMSE
    rmse_list.append([x, y] + rmse_values + [avg_rmse])

# Create the final DataFrame
rmse_df = pd.DataFrame(rmse_list, columns=['x', 'y', 'f_1', 'f_3', 'f_6', 'f_9', 'f_12', 'Avg_RMSE'])

# Save the DataFrame to a CSV file
rmse_df.to_csv("rmse_lstm_rw.csv", index=False)

# Print the RMSE comparison table
print(rmse_df)

            x         y       f_1        f_3       f_6       f_9      f_12  \
0  103.989682  1.356130  3.677622   2.476457  1.825618  5.251563  2.855244   
1  104.006695  1.343977  4.077166   1.028193  4.492235  1.056429  3.996095   
2  104.013177  1.368283  7.787229  16.396805  6.775831  9.652981  9.440830   
3  104.011557  1.319672  0.452054  10.116734  0.240243  6.845947  0.311957   
4  104.020468  1.367472  9.169594  10.616344  2.053777  3.883152  4.841895   
5  104.031001  1.358560  6.501104   6.076711  6.604288  5.371824  3.366160   
6  103.982390  1.393398  8.407915   1.094688  5.110337  0.688941  7.025957   
7  103.990492  1.331014  7.950094   0.296769  5.819171  3.065353  2.857167   
8  103.976719  1.352079  1.255078   4.765434  1.820464  2.532577  5.337869   
9  103.990492  1.346408  3.648005   3.236481  6.054810  2.429450  1.670325   

   Avg_RMSE  
0  4.399817  
1  5.452699  
2  8.092085  
3  6.343703  
4  6.811258  
5  5.257598  
6  5.610250  
7  5.017631  
8  3.560273  
9